Vào lệnh (LONG): Giá Đóng Cửa > EMA200 VÀ RSI < 30. (Mua lúc giá điều chỉnh trong xu hướng tăng).

Dừng lỗ (SL): Đặt tại Giá Mua - (1.5 x ATR). (Cắt lỗ động theo biên độ nến, tránh nhiễu).

Chốt lời (TP): Đóng toàn bộ lệnh khi RSI > 65

In [14]:
# Install required libraries
!pip install pandas pandas-ta requests plotly tabulate -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['BNB-USD', 'BTCUSDT', 'ETHUSDT'],
    'INTERVAL': '1h',
    'START_DATE': '01/01/2020',
    'END_DATE': '01/01/2022',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==========================================
# MODULE 1: DATA HANDLER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.display_symbol = symbol.upper()
        # Convert 'BTC-USD' thành 'BTCUSDT' để tương thích API Binance Spot
        self.api_symbol = self.display_symbol.replace('-USD', 'USDT')
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"

        print(f"Checking connection for {self.api_symbol}...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451: # Blocked IP (US)
                base_url = "https://api.binance.us"
        except Exception:
            base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines = []
        print(f"Downloading {self.display_symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": self.api_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        return df

# ==========================================
# MODULE 2: STRATEGY LOGIC
# ==========================================
class TrendPullbackStrategy:
    def __init__(self):
        self.ema_period = 200
        self.rsi_period = 14
        self.atr_period = 14

        self.rsi_entry = 30
        self.rsi_tp = 65
        self.atr_multiplier = 1.5

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        df['EMA_200'] = ta.ema(df['Close'], length=self.ema_period)
        df['RSI'] = ta.rsi(df['Close'], length=self.rsi_period)
        df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=self.atr_period)
        df.dropna(inplace=True)
        return df

# ==========================================
# MODULE 3: BACKTEST ENGINE
# ==========================================
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.sl_price = 0.0
        self.trades = []

        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        O = self.df['Open'].values
        L = self.df['Low'].values
        C = self.df['Close'].values
        times = self.df.index

        ema200 = self.df['EMA_200'].values
        rsi = self.df['RSI'].values
        atr = self.df['ATR'].values

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']

        s = self.strategy
        self.equity_curve[0] = self.capital

        for i in range(1, len(self.df)):
            # ĐÓNG LỆNH
            if self.position == 1:
                exit_signal = False
                reason = ""
                real_exit = 0.0

                if rsi[i-1] > s.rsi_tp:
                    exit_signal = True
                    reason = "TP (RSI > 65)"
                    real_exit = O[i] * (1 - slip)

                elif L[i] <= self.sl_price:
                    exit_signal = True
                    reason = "Stop Loss (ATR)"
                    real_exit = min(O[i], self.sl_price) * (1 - slip)

                if exit_signal:
                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long', 'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    self.position = 0
                    self.invested_amount = 0.0
                    self.sl_price = 0.0

            # VÀO LỆNH
            if self.position == 0:
                if C[i-1] > ema200[i-1] and rsi[i-1] < s.rsi_entry:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)
                    self.sl_price = self.entry_price - (s.atr_multiplier * atr[i-1])
                    self.invested_amount = self.capital * pos_size_pct

            # TÍNH VỐN
            current_equity = self.capital
            if self.position == 1:
                current_value = self.invested_amount * (C[i] / self.entry_price)
                unrealized_pnl = current_value - self.invested_amount
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==========================================
# MODULE 4: PERFORMANCE EVALUATOR
# ==========================================
class Evaluator:
    @staticmethod
    def evaluate(equity, trades_df, initial_capital, symbol, interval):
        if trades_df.empty:
            return None

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        # Hàm nhận diện số nến trong 1 năm
        def get_annual_factor(tf):
            if tf == '15m': return 365 * 24 * 4
            elif tf == '1h': return 365 * 24
            elif tf == '4h': return 365 * 6
            elif tf == '1d': return 365
            return 365 * 24 # Default 1h

        annual_factor = get_annual_factor(interval)
        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(annual_factor) if returns.std() != 0 else 0

        return {
            'Symbol': symbol,
            'Total Trades': len(trades_df),
            'Winrate': f"{winrate:.2%}",
            'Profit Factor': f"{profit_factor:.2f}",
            'Max Drawdown': f"{max_dd:.2%}",
            'Sharpe Ratio': round(sharpe_ratio, 2),
            'Net PnL ($)': round(net_pnl, 2),
            'Total Return': f"{total_return:.2%}"
        }

    @staticmethod
    def plot_chart(equity, drawdown, symbol, interval):
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.7, 0.3])
        fig.add_trace(go.Scatter(x=equity.index, y=equity.values, mode="lines", name="Equity", line=dict(color="#2962ff", width=1.5)), row=1, col=1)
        fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown.values, mode="lines", name="Drawdown", line=dict(color="#d32f2f", width=1), fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'), row=2, col=1)

        fig.update_layout(title=f"Performance: {symbol} ({interval})", template="plotly_white", height=500, margin=dict(l=40, r=40, t=60, b=40), showlegend=False)
        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", row=1, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", row=2, col=1)
        fig.show()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    summary_results = []

    for symbol in CONFIG['SYMBOLS']:
        print(f"\n[{symbol}] Processing...")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            my_strategy = TrendPullbackStrategy()
            engine = BacktestEngine(df_raw, my_strategy, CONFIG)
            equity, trades = engine.run()

            # Evaluate Metrics
            metrics = Evaluator.evaluate(equity, trades, CONFIG['INITIAL_CAPITAL'], symbol, CONFIG['INTERVAL'])

            if metrics:
                summary_results.append(metrics)

                # Plot Chart
                rolling_max = equity.cummax()
                drawdown = (equity - rolling_max) / rolling_max
                Evaluator.plot_chart(equity, drawdown, symbol, CONFIG['INTERVAL'])
            else:
                print(f"[{symbol}] No trades executed.")

    # IN RA BẢNG TỔNG HỢP (SUMMARY TABLE)
    if summary_results:
        print("\n" + "="*80)
        print(" " * 25 + f"BACKTEST SUMMARY ({CONFIG['INTERVAL']})")
        print("="*80)
        summary_df = pd.DataFrame(summary_results)
        print(summary_df.to_markdown(index=False))
        print("="*80 + "\n")

Output hidden; open in https://colab.research.google.com to view.